# Inspect a Duckiedrone ROS 2 Graph

This is the only notebook in this learning experience (LX) that can require a physical or virtual Duckiedrone. Work only with a Duckiedrone that you own or are authorized to inspect. This workflow is read-only: it observes Robot Operating System 2 (ROS 2) telemetry and graph metadata and does not send flight-control commands, arm a vehicle, or change device configuration.

## Enter the authorized device context

Use the connection workflow from the Linux and Networking learning experience to reach the physical or virtual Duckiedrone. The base-station shell, the Duckiedrone shell, and a service-container shell are different contexts. Once you are on the authorized Duckiedrone, list its running containers:

```bash
docker ps
```

[MAVROS](https://github.com/mavlink/mavros) is a bridge between ROS 2 and the open-source [PX4](https://docs.px4.io/main/) autopilot firmware on the flight controller. It translates [MAVLink](https://mavlink.io/), the flight-controller messaging protocol, between the flight controller and ROS 2 topics, services, and actions. The [current Duckiedrone software stack](https://docs.duckietown.com/ente/opmanual-dd24/) runs that bridge in its `ros2-mavros` container. Open its interactive shell only to inspect the configured graph:

```bash
docker exec -it ros2-mavros bash
```

`docker exec -it` opens a shell inside a running container. The `-it` options keep it attached to your terminal so you can type commands. This container's shell normally prepares the installed ROS 2 environment and retains the middleware and domain settings configured on the Duckiedrone.

## Confirm the ROS 2 context

Before inspection, confirm that the shell has the ROS 2 command and record its existing discovery settings:

```bash
command -v ros2
echo "$ROS_DOMAIN_ID"
echo "$RMW_IMPLEMENTATION"
```

On a Duckiedrone running the current software stack, the final two commands report `42` and `rmw_zenoh_cpp`. If `command -v ros2` prints no path, source the installed distribution before continuing:

```bash
source /opt/ros/${ROS2_DISTRO}/setup.bash
```

Inspect these values; do not change them merely to make another terminal discover the graph. Notebook 4 explains why compatible middleware and `ROS_DOMAIN_ID` matter.

## Read the graph

Now use only read-only ROS 2 inspection commands:

```bash
ros2 node list
ros2 topic list -t
ros2 topic info -v /mavros/imu/data
```

The final command reports the message type, publishers, subscriptions, and Quality of Service (QoS) settings, the rules that control message delivery. Use the observed type and QoS profile when a later exercise creates a subscriber rather than assuming that every sensor publishes with the same settings.

On a Duckiedrone named `ROBOT_NAME`, camera and bottom time-of-flight (ToF) sensor bridge topics commonly appear as `/ROBOT_NAME/camera_node/image/compressed` and `/ROBOT_NAME/bottom_tof_driver_node/range`. MAVROS is a separate root-level path, so `/mavros/imu/data` and `/mavros/state` do not have that prefix. The running stack and enabled plugins decide the final list; always inspect it before writing a subscription.

The Dashboard is another consumer of this ROS 2 graph. Its `ros2-rosbridge-websocket` service translates ROS 2 traffic for the browser, so a Dashboard display does not create a separate sensor-data path.

Leave the container shell with `exit` when you finish. Do not use this notebook to publish commands, call services or actions, modify parameters, or control flight hardware.

## Further reading

The ROS 2 Jazzy documentation on [Quality of Service settings](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Quality-of-Service-Settings.html) explains the connection details shown by `ros2 topic info -v`.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
